In [2]:
# STEP 3 - PySpark Pipeline
# Create: processed/spark_analysis.py
'''
Repeat ALL four tasks from Step 2
using PySpark.

Use:
- spark.read.csv() for loading
- withColumn() + when() for new columns
- df.join() for merging
- groupBy().agg() for analysis
- df.write.csv() for saving output

Save PySpark outputs to:
reports/spark_dept_summary.csv
reports/spark_band_distribution.csv
reports/spark_top_earners.csv
reports/spark_city_summary.csv
'''

'\nRepeat ALL four tasks from Step 2\nusing PySpark.\n\nUse:\n- spark.read.csv() for loading\n- withColumn() + when() for new columns\n- df.join() for merging\n- groupBy().agg() for analysis\n- df.write.csv() for saving output\n\nSave PySpark outputs to:\nreports/spark_dept_summary.csv\nreports/spark_band_distribution.csv\nreports/spark_top_earners.csv\nreports/spark_city_summary.csv\n'

In [3]:
# Task 1 — Load and profile:

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("pyspark_pipeline") \
    .getOrCreate()

In [4]:
emp_df = spark.read.csv("../raw/employees.csv", header=True, inferSchema=True)
dept_df = spark.read.csv("../raw/departments.csv", header=True, inferSchema=True)

In [5]:
emp_df.show()
dept_df.show()

+------+-----+-----------+------+-----+--------+
|emp_id| name|       dept|salary|years|    city|
+------+-----+-----------+------+-----+--------+
|     1|Alice|Engineering| 85000|    5|New York|
|     2|  Bob|  Marketing| 62000|    3| Chicago|
|     3|Carol|Engineering| 92000|    8|New York|
|     4|David|         HR| 55000|    2|  Dallas|
|     5|  Eva|    Finance| 78000|    6|New York|
|     6|Frank|Engineering| 88000|    7| Chicago|
|     7|Grace|    Finance| 67000|    4|  Dallas|
|     8|Henry|         HR| 71000|    5|New York|
|     9|  Ian|  Marketing| 58000|    2| Chicago|
|    10| Jane|Engineering| 95000|    9|New York|
+------+-----+-----------+------+-----+--------+

+-----------+-------+--------+
|       dept| budget|location|
+-----------+-------+--------+
|Engineering|1500000|New York|
|  Marketing| 800000| Chicago|
|    Finance| 600000|New York|
|         HR| 400000|  Dallas|
|      Legal| 350000|New York|
+-----------+-------+--------+



In [6]:
(emp_df.count(), len(emp_df.columns))

(10, 6)

In [7]:
emp_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- years: integer (nullable = true)
 |-- city: string (nullable = true)



In [8]:
from pyspark.sql.functions import *

emp_df.filter(col("salary").isNull()).show()

+------+----+----+------+-----+----+
|emp_id|name|dept|salary|years|city|
+------+----+----+------+-----+----+
+------+----+----+------+-----+----+



In [9]:
dept_df.filter(col("budget").isNull()).show()

+----+------+--------+
|dept|budget|location|
+----+------+--------+
+----+------+--------+



In [10]:
# emp_df.select("name", trim("name").alias("trimed_name"), length("trimed_name")).show()

In [11]:
from pyspark.sql.types import StringType

# trim all string columns
emp_df.select([
    trim(col(c)).alias(c) if isinstance(emp_df.schema[c].dataType, StringType)
    else col(c)
    for c in emp_df.columns
]).show()

+------+-----+-----------+------+-----+--------+
|emp_id| name|       dept|salary|years|    city|
+------+-----+-----------+------+-----+--------+
|     1|Alice|Engineering| 85000|    5|New York|
|     2|  Bob|  Marketing| 62000|    3| Chicago|
|     3|Carol|Engineering| 92000|    8|New York|
|     4|David|         HR| 55000|    2|  Dallas|
|     5|  Eva|    Finance| 78000|    6|New York|
|     6|Frank|Engineering| 88000|    7| Chicago|
|     7|Grace|    Finance| 67000|    4|  Dallas|
|     8|Henry|         HR| 71000|    5|New York|
|     9|  Ian|  Marketing| 58000|    2| Chicago|
|    10| Jane|Engineering| 95000|    9|New York|
+------+-----+-----------+------+-----+--------+



In [12]:
# trim all string columns
dept_df.select([
    trim(col(c)).alias(c) if isinstance(dept_df.schema[c].dataType, StringType)
    else col(c)
    for c in dept_df.columns
]).show()

+-----------+-------+--------+
|       dept| budget|location|
+-----------+-------+--------+
|Engineering|1500000|New York|
|  Marketing| 800000| Chicago|
|    Finance| 600000|New York|
|         HR| 400000|  Dallas|
|      Legal| 350000|New York|
+-----------+-------+--------+



In [13]:
emp_df = emp_df.withColumn(
    "salary_band",
    when(col("salary") >= 85000, "Senior")
    .when(col("salary") >= 65000, "Mid")
    .otherwise("Junior") 
)

In [14]:
emp_df = emp_df.withColumn(
    "experience_band",
    when(col("years") >= 7, "Veteran")
    .when(col("years") >= 4, "Experienced")
    .otherwise("Fresher")
)

In [15]:
emp_df.show()

+------+-----+-----------+------+-----+--------+-----------+---------------+
|emp_id| name|       dept|salary|years|    city|salary_band|experience_band|
+------+-----+-----------+------+-----+--------+-----------+---------------+
|     1|Alice|Engineering| 85000|    5|New York|     Senior|    Experienced|
|     2|  Bob|  Marketing| 62000|    3| Chicago|     Junior|        Fresher|
|     3|Carol|Engineering| 92000|    8|New York|     Senior|        Veteran|
|     4|David|         HR| 55000|    2|  Dallas|     Junior|        Fresher|
|     5|  Eva|    Finance| 78000|    6|New York|        Mid|    Experienced|
|     6|Frank|Engineering| 88000|    7| Chicago|     Senior|        Veteran|
|     7|Grace|    Finance| 67000|    4|  Dallas|        Mid|    Experienced|
|     8|Henry|         HR| 71000|    5|New York|        Mid|    Experienced|
|     9|  Ian|  Marketing| 58000|    2| Chicago|     Junior|        Fresher|
|    10| Jane|Engineering| 95000|    9|New York|     Senior|        Veteran|

In [16]:
# Task 3 — Merge with departments:

result = emp_df.join(dept_df, on="dept", how="left")

In [17]:
#result.write.csv("../processed/sp_enriched_employees.csv", header=True)

In [18]:
# Task 4 — Analysis (save each result):

dept_summary = result.groupBy("dept").agg(
    count("emp_id").alias("Headcount"),
    mean("salary").alias("avg_salary"),
    sum("salary").alias("total_salary"),
    sum("budget").alias("total_budget"),
    first("budget").alias("budget")
)

In [19]:
dept_summary = dept_summary.withColumn(
    "variance", 
    col("budget") - col("total_salary")
).show()

+-----------+---------+----------+------------+------------+-------+--------+
|       dept|Headcount|avg_salary|total_salary|total_budget| budget|variance|
+-----------+---------+----------+------------+------------+-------+--------+
|Engineering|        4|   90000.0|      360000|     6000000|1500000| 1140000|
|         HR|        2|   63000.0|      126000|      800000| 400000|  274000|
|    Finance|        2|   72500.0|      145000|     1200000| 600000|  455000|
|  Marketing|        2|   60000.0|      120000|     1600000| 800000|  680000|
+-----------+---------+----------+------------+------------+-------+--------+



In [20]:
dept_summary.write.csv("processed", header=True)

AttributeError: 'NoneType' object has no attribute 'write'

In [ ]:
# - Salary band distribution:
#   count per band.
#   Save to reports/band_distribution.csv

band_distribution = emp_df.groupBy("salary_band").agg(
    count("emp_id").alias("emp_counts")
).show()

+-----------+----------+
|salary_band|emp_counts|
+-----------+----------+
|     Senior|         4|
|        Mid|         3|
|     Junior|         3|
+-----------+----------+



In [33]:
# - Top 3 earners per department.
#   Save to reports/top_earners.csv

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("dept") \
                    .orderBy(col("salary").desc())

top_earners = emp_df.withColumn(
    "rank", 
    row_number().over(window_spec)
).filter(col("rank") == 1)

In [35]:
top_earners.show(3)

+------+-----+-----------+------+-----+--------+-----------+---------------+----+
|emp_id| name|       dept|salary|years|    city|salary_band|experience_band|rank|
+------+-----+-----------+------+-----+--------+-----------+---------------+----+
|    10| Jane|Engineering| 95000|    9|New York|     Senior|        Veteran|   1|
|     5|  Eva|    Finance| 78000|    6|New York|        Mid|    Experienced|   1|
|     8|Henry|         HR| 71000|    5|New York|        Mid|    Experienced|   1|
+------+-----+-----------+------+-----+--------+-----------+---------------+----+
only showing top 3 rows


In [41]:
# - City-wise headcount and avg salary.
#   Save to reports/city_summary.csv

city_wise_summary = emp_df.groupBy("city").agg(
    count("emp_id").alias("emp_count"),
    round(avg("salary"), 2).alias("avg_salary")
).show()

+--------+---------+----------+
|    city|emp_count|avg_salary|
+--------+---------+----------+
|  Dallas|        2|   61000.0|
| Chicago|        3|  69333.33|
|New York|        5|   84200.0|
+--------+---------+----------+

